In [4]:
pip install PyQt5 pandas numpy scikit-learn matplotlib scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
"""
Docking Validation Toolkit v2.0 - Customized for Your Data
Specialized for files with: Ligand, Affinity (kcal/mol) columns
Author: Virtual Screening Pipeline
Version: 2.1.0
"""

import sys
import os
import pandas as pd
import numpy as np
import matplotlib
# Set the matplotlib backend to Qt5Agg BEFORE importing pyplot
matplotlib.use('Qt5Agg')

from PyQt5.QtWidgets import (QApplication, QMainWindow, QWidget, QVBoxLayout, 
                            QHBoxLayout, QLabel, QPushButton, QFileDialog, 
                            QTextEdit, QTabWidget, QTableWidget, QTableWidgetItem,
                            QGroupBox, QGridLayout, QComboBox, QSpinBox, 
                            QDoubleSpinBox, QCheckBox, QMessageBox, QProgressBar,
                            QSplitter, QListWidget, QListWidgetItem, QFrame,
                            QLineEdit, QFormLayout, QDialog, QDialogButtonBox,
                            QHeaderView)
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QTimer
from PyQt5.QtGui import QFont, QColor, QPalette, QIcon
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from sklearn.metrics import roc_auc_score, roc_curve, auc
from scipy import stats
import warnings
import re
warnings.filterwarnings('ignore')

# ==================== METRICS CALCULATION FUNCTIONS ====================

class ValidationMetrics:
    """Class to calculate all validation metrics"""
    
    @staticmethod
    def calculate_roc_auc(scores, labels, lower_is_better=True):
        """Calculate ROC-AUC and related metrics"""
        # For AutoDock Vina: lower docking score = better
        if lower_is_better:
            scores_for_roc = -np.array(scores)
        else:
            scores_for_roc = np.array(scores)
        
        labels = np.array(labels)
        
        # Check if we have both classes
        if len(np.unique(labels)) < 2:
            return {
                'ROC_AUC': 0.5,
                'ROC_AUC_CI': (0.5, 0.5),
                'ROC_AUC_SE': 0.0,
                'Sensitivity': 0.0,
                'Specificity': 0.0,
                'FPR': np.array([0, 1]),
                'TPR': np.array([0, 1])
            }
        
        try:
            roc_auc = roc_auc_score(labels, scores_for_roc)
        except:
            roc_auc = 0.5
        
        # Calculate ROC curve
        fpr, tpr, thresholds = roc_curve(labels, scores_for_roc)
        
        # Calculate Youden's J statistic for optimal threshold
        if len(fpr) > 0 and len(tpr) > 0:
            youden_j = tpr - fpr
            optimal_idx = np.argmax(youden_j)
            sensitivity = tpr[optimal_idx]
            specificity = 1 - fpr[optimal_idx]
        else:
            sensitivity = 0.0
            specificity = 0.0
        
        # DeLong confidence interval
        ci_lower, ci_upper, se_auc = ValidationMetrics._delong_ci(labels, scores_for_roc)
        
        return {
            'ROC_AUC': roc_auc,
            'ROC_AUC_CI': (ci_lower, ci_upper),
            'ROC_AUC_SE': se_auc,
            'Sensitivity': sensitivity,
            'Specificity': specificity,
            'FPR': fpr,
            'TPR': tpr
        }
    
    @staticmethod
    def _delong_ci(y_true, y_score, alpha=0.95):
        """Calculate DeLong confidence interval for ROC-AUC"""
        n = len(y_true)
        n1 = sum(y_true == 1)
        n0 = n - n1
        
        if n1 == 0 or n0 == 0:
            return 0.5, 0.5, 0.0
        
        try:
            scores_1 = y_score[y_true == 1]
            scores_0 = y_score[y_true == 0]
            
            V10 = []
            V01 = []
            
            for i in range(n1):
                v = np.sum(scores_1[i] > scores_0) + 0.5 * np.sum(scores_1[i] == scores_0)
                V10.append(v / n0)
            
            for j in range(n0):
                v = np.sum(scores_0[j] < scores_1) + 0.5 * np.sum(scores_0[j] == scores_1)
                V01.append(v / n1)
            
            V10 = np.array(V10)
            V01 = np.array(V01)
            
            S10 = np.var(V10, ddof=1) if len(V10) > 1 else 0
            S01 = np.var(V01, ddof=1) if len(V01) > 1 else 0
            
            SE = np.sqrt(S10 / n1 + S01 / n0)
            
            z = stats.norm.ppf((1 + alpha) / 2)
            
            auc_val = roc_auc_score(y_true, y_score)
            ci_lower = max(0, auc_val - z * SE)
            ci_upper = min(1, auc_val + z * SE)
            
            return ci_lower, ci_upper, SE
        except:
            return 0.5, 0.5, 0.0
    
    @staticmethod
    def calculate_enrichment_factors(scores, labels, fractions=[0.01, 0.05, 0.1], lower_is_better=True):
        """Calculate Enrichment Factors at specified fractions"""
        # Sort by score
        if lower_is_better:
            rank_scores = -np.array(scores)
        else:
            rank_scores = np.array(scores)
            
        labels = np.array(labels)
        
        # Sort indices by rank_score
        sorted_indices = np.argsort(rank_scores)[::-1]  # Descending
        sorted_labels = labels[sorted_indices]
        
        n_total = len(labels)
        n_actives = np.sum(labels)
        
        if n_actives == 0 or n_total == 0:
            return {f'EF_{int(fraction*100)}%': 0 for fraction in fractions}
        
        results = {}
        
        for fraction in fractions:
            n_top = max(1, int(n_total * fraction))
            n_actives_top = np.sum(sorted_labels[:n_top])
            
            expected_actives = n_top * (n_actives / n_total)
            
            if expected_actives > 0:
                ef = n_actives_top / expected_actives
            else:
                ef = 0
            
            hit_rate = n_actives_top / n_top if n_top > 0 else 0
            yield_fraction = n_actives_top / n_actives if n_actives > 0 else 0
            
            results[f'EF_{int(fraction*100)}%'] = ef
            results[f'HR_{int(fraction*100)}%'] = hit_rate
            results[f'Yield_{int(fraction*100)}%'] = yield_fraction
            results[f'Actives_{int(fraction*100)}%'] = n_actives_top
        
        return results
    
    @staticmethod
    def calculate_bedroc(scores, labels, alpha=20.0, lower_is_better=True):
        """Calculate BEDROC (Boltzmann-Enhanced Discrimination of ROC)"""
        if lower_is_better:
            rank_scores = -np.array(scores)
        else:
            rank_scores = np.array(scores)
            
        labels = np.array(labels)
        
        sorted_indices = np.argsort(rank_scores)[::-1]
        sorted_labels = labels[sorted_indices]
        
        n_total = len(labels)
        n_actives = np.sum(labels)
        
        if n_actives == 0 or n_actives == n_total or n_total == 0:
            return {'BEDROC': 0, 'BEDROC_norm': 0, 'RIE': 0, 'alpha': alpha}
        
        # Get ranks of actives (1-based)
        active_ranks = []
        for i, label in enumerate(sorted_labels):
            if label == 1:
                active_ranks.append(i + 1)
        
        if len(active_ranks) == 0:
            return {'BEDROC': 0, 'BEDROC_norm': 0, 'RIE': 0, 'alpha': alpha}
        
        active_ranks = np.array(active_ranks)
        
        # Calculate BEDROC
        sum_exp = np.sum(np.exp(-alpha * active_ranks / n_total))
        
        # Calculate Rmax (maximum possible sum)
        if alpha == 0:
            Rmax = n_actives
        else:
            Rmax = (1 - np.exp(-alpha)) / (np.exp(alpha/n_total) - 1)
        
        # Calculate Rmin (minimum possible sum)
        Rmin = np.sum(np.exp(-alpha * np.arange(1, n_actives + 1) / n_total))
        
        bedroc = sum_exp / Rmax
        bedroc_norm = (sum_exp - Rmin) / (Rmax - Rmin) if Rmax != Rmin else 0
        
        # Calculate RIE
        rie = sum_exp / (n_actives / n_total * (1 - np.exp(-alpha)) / (np.exp(alpha/n_total) - 1))
        
        return {
            'BEDROC': bedroc,
            'BEDROC_norm': bedroc_norm,
            'RIE': rie,
            'alpha': alpha
        }
    
    @staticmethod
    def calculate_statistical_tests(active_scores, decoy_scores):
        """Calculate statistical significance tests"""
        if len(active_scores) == 0 or len(decoy_scores) == 0:
            return {
                'MannWhitney_p': 1.0,
                'T_test_p': 1.0,
                'KS_p': 1.0,
                'Cohens_d': 0.0,
                'Active_mean': 0.0,
                'Active_std': 0.0,
                'Decoy_mean': 0.0,
                'Decoy_std': 0.0
            }
        
        # Mann-Whitney U test
        try:
            stat_mw, p_mw = stats.mannwhitneyu(active_scores, decoy_scores, alternative='less')
        except:
            p_mw = 1.0
        
        # Student's t-test
        try:
            stat_t, p_t = stats.ttest_ind(active_scores, decoy_scores, equal_var=False)
        except:
            p_t = 1.0
        
        # Kolmogorov-Smirnov test
        try:
            stat_ks, p_ks = stats.ks_2samp(active_scores, decoy_scores)
        except:
            p_ks = 1.0
        
        # Cohen's d
        try:
            n1, n2 = len(active_scores), len(decoy_scores)
            var1, var2 = np.var(active_scores, ddof=1), np.var(decoy_scores, ddof=1)
            pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
            cohen_d = (np.mean(active_scores) - np.mean(decoy_scores)) / pooled_std
        except:
            cohen_d = 0.0
        
        return {
            'MannWhitney_p': p_mw,
            'T_test_p': p_t,
            'KS_p': p_ks,
            'Cohens_d': cohen_d,
            'Active_mean': np.mean(active_scores) if len(active_scores) > 0 else 0.0,
            'Active_std': np.std(active_scores) if len(active_scores) > 0 else 0.0,
            'Decoy_mean': np.mean(decoy_scores) if len(decoy_scores) > 0 else 0.0,
            'Decoy_std': np.std(decoy_scores) if len(decoy_scores) > 0 else 0.0
        }
    
    @staticmethod
    def calculate_additional_metrics(scores, labels, lower_is_better=True):
        """Calculate additional metrics"""
        # Sort by score
        if lower_is_better:
            sorted_indices = np.argsort(scores)  # Ascending (lower score = better)
        else:
            sorted_indices = np.argsort(scores)[::-1]  # Descending (higher score = better)
            
        sorted_labels = labels[sorted_indices]
        
        n_total = len(labels)
        n_actives = np.sum(labels)
        
        if n_actives == 0 or n_total == 0:
            return {
                'Best_Active_Rank': np.nan,
                'Worst_Active_Rank': np.nan,
                'Avg_Active_Rank': np.nan,
                'Median_Active_Rank': np.nan,
                'Active_Fraction': 0.0
            }
        
        # Find ranks of actives
        active_ranks = []
        for i, label in enumerate(sorted_labels):
            if label == 1:
                active_ranks.append(i + 1)  # 1-based rank
        
        if len(active_ranks) == 0:
            return {
                'Best_Active_Rank': np.nan,
                'Worst_Active_Rank': np.nan,
                'Avg_Active_Rank': np.nan,
                'Median_Active_Rank': np.nan,
                'Active_Fraction': 0.0
            }
        
        best_active_rank = min(active_ranks)
        worst_active_rank = max(active_ranks)
        avg_active_rank = np.mean(active_ranks)
        median_active_rank = np.median(active_ranks)
        
        # Calculate GH score
        def calculate_gh_score(fraction=0.01):
            n_top = max(1, int(n_total * fraction))
            n_actives_top = np.sum(sorted_labels[:n_top])
            
            Ha = n_actives_top
            Ht = n_top
            A = n_actives
            T = n_total
            
            if Ht > 0 and A > 0 and T > 0:
                gh = (Ha/Ht) * (Ha/A) / np.sqrt(Ht/T)
            else:
                gh = 0
            return gh
        
        gh_1 = calculate_gh_score(0.01)
        gh_5 = calculate_gh_score(0.05)
        
        return {
            'Best_Active_Rank': best_active_rank,
            'Worst_Active_Rank': worst_active_rank,
            'Avg_Active_Rank': avg_active_rank,
            'Median_Active_Rank': median_active_rank,
            'GH_Score_1%': gh_1,
            'GH_Score_5%': gh_5,
            'Active_Fraction': n_actives / n_total if n_total > 0 else 0.0
        }

# ==================== COLUMN MAPPING DIALOG ====================

class ColumnMappingDialog(QDialog):
    """Dialog for mapping CSV columns - Customized for your specific files"""
    
    def __init__(self, actives_columns, decoys_columns, gene_name="", parent=None):
        super().__init__(parent)
        self.setWindowTitle("Confirm Column Mapping for Your Files")
        self.setGeometry(300, 300, 500, 400)
        
        layout = QVBoxLayout()
        
        # Gene name
        gene_layout = QHBoxLayout()
        gene_layout.addWidget(QLabel("Gene/Target Name:"))
        self.gene_edit = QLineEdit(gene_name)
        self.gene_edit.setPlaceholderText("e.g., EGFR, AKT1, COVID-19_Mpro")
        gene_layout.addWidget(self.gene_edit)
        layout.addLayout(gene_layout)
        
        # Information about detected columns
        info_label = QLabel(f"Detected columns in Actives file: {', '.join(actives_columns)}\n"
                           f"Detected columns in Decoys file: {', '.join(decoys_columns)}")
        info_label.setWordWrap(True)
        info_label.setStyleSheet("background-color: #f0f8ff; padding: 10px; border-radius: 5px;")
        layout.addWidget(info_label)
        
        # Actives column mapping
        actives_group = QGroupBox("Actives CSV Columns")
        actives_layout = QFormLayout()
        
        self.actives_id_combo = QComboBox()
        self.actives_id_combo.addItems(actives_columns)
        actives_layout.addRow("Ligand ID Column:", self.actives_id_combo)
        
        self.actives_score_combo = QComboBox()
        self.actives_score_combo.addItems(actives_columns)
        actives_layout.addRow("Affinity Score Column:", self.actives_score_combo)
        
        actives_group.setLayout(actives_layout)
        layout.addWidget(actives_group)
        
        # Decoys column mapping
        decoys_group = QGroupBox("Decoys CSV Columns")
        decoys_layout = QFormLayout()
        
        self.decoys_id_combo = QComboBox()
        self.decoys_id_combo.addItems(decoys_columns)
        decoys_layout.addRow("Ligand ID Column:", self.decoys_id_combo)
        
        self.decoys_score_combo = QComboBox()
        self.decoys_score_combo.addItems(decoys_columns)
        decoys_layout.addRow("Affinity Score Column:", self.decoys_score_combo)
        
        decoys_group.setLayout(decoys_layout)
        layout.addWidget(decoys_group)
        
        # Auto-detect based on your specific column names
        self.auto_detect_for_your_files(actives_columns, decoys_columns)
        
        # Buttons
        button_box = QDialogButtonBox(QDialogButtonBox.Ok | QDialogButtonBox.Cancel)
        button_box.accepted.connect(self.accept)
        button_box.rejected.connect(self.reject)
        layout.addWidget(button_box)
        
        self.setLayout(layout)
    
    def auto_detect_for_your_files(self, actives_cols, decoys_cols):
        """Auto-detect column names specifically for your files"""
        # Based on your file structure: "Ligand" and "Affinity (kcal/mol)"
        
        # For Ligand ID column
        ligand_options = ['Ligand', 'ligand', 'LIGAND', 'Name', 'name', 'NAME', 'ID', 'Id', 'id']
        for option in ligand_options:
            if option in actives_cols:
                self.actives_id_combo.setCurrentText(option)
                break
            # Try case-insensitive
            for col in actives_cols:
                if col.lower() == option.lower():
                    self.actives_id_combo.setCurrentText(col)
                    break
        
        # For Affinity score column
        score_options = ['Affinity (kcal/mol)', 'Affinity', 'affinity', 'docking_score', 'score', 'binding_energy']
        for option in score_options:
            if option in actives_cols:
                self.actives_score_combo.setCurrentText(option)
                break
            # Try partial match
            for col in actives_cols:
                if 'affinity' in col.lower() or 'kcal' in col.lower():
                    self.actives_score_combo.setCurrentText(col)
                    break
        
        # Same for decoys
        for option in ligand_options:
            if option in decoys_cols:
                self.decoys_id_combo.setCurrentText(option)
                break
            for col in decoys_cols:
                if col.lower() == option.lower():
                    self.decoys_id_combo.setCurrentText(col)
                    break
        
        for option in score_options:
            if option in decoys_cols:
                self.decoys_score_combo.setCurrentText(option)
                break
            for col in decoys_cols:
                if 'affinity' in col.lower() or 'kcal' in col.lower():
                    self.decoys_score_combo.setCurrentText(col)
                    break
    
    def get_mapping(self):
        """Get the column mapping"""
        return {
            'gene_name': self.gene_edit.text().strip(),
            'actives_id': self.actives_id_combo.currentText(),
            'actives_score': self.actives_score_combo.currentText(),
            'decoys_id': self.decoys_id_combo.currentText(),
            'decoys_score': self.decoys_score_combo.currentText()
        }

# ==================== WORKER THREAD FOR CALCULATIONS ====================

class ValidationWorker(QThread):
    """Worker thread for validation calculations - Customized for your data"""
    
    progress = pyqtSignal(int)
    result = pyqtSignal(dict)
    error = pyqtSignal(str)
    
    def __init__(self, actives_file, decoys_file, column_mapping, settings):
        super().__init__()
        self.actives_file = actives_file
        self.decoys_file = decoys_file
        self.column_mapping = column_mapping
        self.settings = settings
    
    def run(self):
        try:
            # Load data
            self.progress.emit(10)
            df = self.load_and_merge_data()
            
            if df is None or len(df) == 0:
                self.error.emit("Failed to load data or no valid data found. Check for NA values.")
                return
            
            self.progress.emit(30)
            
            # Extract scores and labels
            scores = df['docking_score'].values
            labels = df['label'].values
            
            # Active and decoy scores
            active_scores = df[df['label'] == 1]['docking_score'].values
            decoy_scores = df[df['label'] == 0]['docking_score'].values
            
            self.progress.emit(50)
            
            # Get settings
            lower_is_better = self.settings.get('lower_is_better', True)
            ef_fractions = self.settings.get('ef_fractions', [0.01, 0.05, 0.10])
            bedroc_alpha = self.settings.get('bedroc_alpha', 20.0)
            
            # Calculate metrics
            roc_metrics = ValidationMetrics.calculate_roc_auc(scores, labels, lower_is_better)
            self.progress.emit(60)
            
            ef_metrics = ValidationMetrics.calculate_enrichment_factors(scores, labels, ef_fractions, lower_is_better)
            self.progress.emit(70)
            
            bedroc_metrics = ValidationMetrics.calculate_bedroc(scores, labels, bedroc_alpha, lower_is_better)
            self.progress.emit(80)
            
            stats_metrics = ValidationMetrics.calculate_statistical_tests(active_scores, decoy_scores)
            self.progress.emit(90)
            
            additional_metrics = ValidationMetrics.calculate_additional_metrics(scores, labels, lower_is_better)
            self.progress.emit(95)
            
            # Combine results
            results = {
                'dataframe': df,
                'scores': scores,
                'labels': labels,
                'active_scores': active_scores,
                'decoy_scores': decoy_scores,
                'gene_name': self.column_mapping['gene_name'],
                'lower_is_better': lower_is_better,
                'ef_fractions': ef_fractions,
                'bedroc_alpha': bedroc_alpha,
                **roc_metrics,
                **ef_metrics,
                **bedroc_metrics,
                **stats_metrics,
                **additional_metrics,
                'n_actives': len(active_scores),
                'n_decoys': len(decoy_scores),
                'n_total': len(scores)
            }
            
            self.progress.emit(100)
            self.result.emit(results)
            
        except Exception as e:
            import traceback
            error_details = traceback.format_exc()
            self.error.emit(f"Error in validation: {str(e)}\n\nDetails:\n{error_details}")
    
    def load_and_merge_data(self):
        """Load and merge active and decoy data with column mapping - HANDLES NA VALUES"""
        try:
            # Read CSV files with NA handling
            actives_df = pd.read_csv(self.actives_file, na_values=['NA', 'N/A', 'na', 'n/a', '', ' ', 'NaN', 'nan'])
            decoys_df = pd.read_csv(self.decoys_file, na_values=['NA', 'N/A', 'na', 'n/a', '', ' ', 'NaN', 'nan'])
            
            print(f"Actives loaded: {len(actives_df)} rows")
            print(f"Decoys loaded: {len(decoys_df)} rows")
            
            # Extract columns using mapping
            actives_id_col = self.column_mapping['actives_id']
            actives_score_col = self.column_mapping['actives_score']
            decoys_id_col = self.column_mapping['decoys_id']
            decoys_score_col = self.column_mapping['decoys_score']
            
            print(f"Using columns - Actives ID: {actives_id_col}, Score: {actives_score_col}")
            print(f"Using columns - Decoys ID: {decoys_id_col}, Score: {decoys_score_col}")
            
            # Check if columns exist
            if actives_id_col not in actives_df.columns:
                raise ValueError(f"Column '{actives_id_col}' not found in Actives file")
            if actives_score_col not in actives_df.columns:
                raise ValueError(f"Column '{actives_score_col}' not found in Actives file")
            if decoys_id_col not in decoys_df.columns:
                raise ValueError(f"Column '{decoys_id_col}' not found in Decoys file")
            if decoys_score_col not in decoys_df.columns:
                raise ValueError(f"Column '{decoys_score_col}' not found in Decoys file")
            
            # Create actives dataframe with proper handling
            actives_processed = pd.DataFrame({
                'compound_id': actives_df[actives_id_col].astype(str),
                'docking_score': pd.to_numeric(actives_df[actives_score_col], errors='coerce'),
                'label': 1
            })
            
            # Create decoys dataframe with proper handling
            decoys_processed = pd.DataFrame({
                'compound_id': decoys_df[decoys_id_col].astype(str),
                'docking_score': pd.to_numeric(decoys_df[decoys_score_col], errors='coerce'),
                'label': 0
            })
            
            print(f"Actives after conversion: {len(actives_processed)} rows")
            print(f"Decoys after conversion: {len(decoys_processed)} rows")
            
            # Count NA values before dropping
            actives_na = actives_processed['docking_score'].isna().sum()
            decoys_na = decoys_processed['docking_score'].isna().sum()
            
            print(f"NA values in Actives: {actives_na}")
            print(f"NA values in Decoys: {decoys_na}")
            
            # Drop NaN scores
            actives_processed = actives_processed.dropna(subset=['docking_score'])
            decoys_processed = decoys_processed.dropna(subset=['docking_score'])
            
            print(f"Actives after dropping NA: {len(actives_processed)} rows")
            print(f"Decoys after dropping NA: {len(decoys_processed)} rows")
            
            # Check if we have data
            if len(actives_processed) == 0:
                raise ValueError("No valid docking scores found in Actives file after removing NA values")
            if len(decoys_processed) == 0:
                raise ValueError("No valid docking scores found in Decoys file after removing NA values")
            
            # Combine
            df = pd.concat([actives_processed, decoys_processed], ignore_index=True)
            
            # Reset index
            df = df.reset_index(drop=True)
            
            # Add original data for reference
            df['gene'] = self.column_mapping['gene_name']
            
            print(f"Final combined dataset: {len(df)} rows")
            print(f"Actives in final dataset: {(df['label'] == 1).sum()}")
            print(f"Decoys in final dataset: {(df['label'] == 0).sum()}")
            
            return df
            
        except Exception as e:
            print(f"Error loading data: {e}")
            import traceback
            traceback.print_exc()
            return None

# ==================== PLOTTING FUNCTIONS ====================

class PlotCanvas(FigureCanvas):
    """Enhanced Matplotlib canvas for embedding plots in PyQt with publication-quality fonts"""
    
    def __init__(self, parent=None, width=8, height=6.5, dpi=100):
        self.fig, self.ax = plt.subplots(figsize=(width, height), dpi=dpi)
        super().__init__(self.fig)
        self.setParent(parent)
        
        # Set default font sizes for publication quality
        self.title_fontsize = 18
        self.label_fontsize = 16
        self.tick_fontsize = 14
        self.legend_fontsize = 14
        self.annotation_fontsize = 12
    
    def plot_roc_curve(self, fpr, tpr, roc_auc, gene_name=""):
        """Plot ROC curve with enhanced fonts"""
        self.ax.clear()
        
        # Plot ROC curve
        self.ax.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})', 
                    linewidth=3, color='blue')
        self.ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random', linewidth=2)
        
        # Set labels with larger fonts
        self.ax.set_xlabel('False Positive Rate', fontsize=self.label_fontsize, fontweight='bold')
        self.ax.set_ylabel('True Positive Rate', fontsize=self.label_fontsize, fontweight='bold')
        
        # Set title
        if gene_name:
            self.ax.set_title(f'ROC Curve - {gene_name}', fontsize=self.title_fontsize, fontweight='bold', pad=20)
        else:
            self.ax.set_title('ROC Curve', fontsize=self.title_fontsize, fontweight='bold', pad=20)
            
        # Configure legend
        self.ax.legend(loc='lower right', fontsize=self.legend_fontsize, framealpha=0.9)
        
        # Configure grid
        self.ax.grid(True, alpha=0.3, linestyle='--')
        
        # Set axis limits
        self.ax.set_xlim([-0.05, 1.05])
        self.ax.set_ylim([-0.05, 1.05])
        
        # Set tick font size
        self.ax.tick_params(axis='both', which='major', labelsize=self.tick_fontsize)
        '''
        # Add AUC annotation
        self.ax.text(0.6, 0.1, f'AUC = {roc_auc:.3f}', 
                    fontsize=self.annotation_fontsize, 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.8))
        
        self.fig.tight_layout()
        self.draw()
    '''
    def plot_score_distribution(self, active_scores, decoy_scores, gene_name="", lower_is_better=True):
        """Plot score distribution histogram with enhanced fonts"""
        self.ax.clear()
        
        # Create histogram
        bins = 30
        alpha = 0.7
        
        # Determine range for bins
        all_scores = np.concatenate([active_scores, decoy_scores])
        hist_range = (np.min(all_scores) - 1, np.max(all_scores) + 1)
        
        # Plot histograms
        self.ax.hist(active_scores, bins=bins, alpha=alpha, label='Actives', 
                    density=True, color='blue', range=hist_range, edgecolor='black', linewidth=0.5)
        self.ax.hist(decoy_scores, bins=bins, alpha=alpha, label='Decoys', 
                    density=True, color='red', range=hist_range, edgecolor='black', linewidth=0.5)
        
        score_type = "lower is better" if lower_is_better else "higher is better"
        
        # Set title and labels with larger fonts
        if gene_name:
            self.ax.set_title(f'Score Distribution - {gene_name}', fontsize=self.title_fontsize, fontweight='bold', pad=20)
        else:
            self.ax.set_title('Score Distribution', fontsize=self.title_fontsize, fontweight='bold', pad=20)
            
        self.ax.set_xlabel(f'Docking Score ({score_type})', fontsize=self.label_fontsize, fontweight='bold')
        self.ax.set_ylabel('Density', fontsize=self.label_fontsize, fontweight='bold')
        
        # Configure legend
        self.ax.legend(fontsize=self.legend_fontsize, framealpha=0.9)
        
        # Configure grid
        self.ax.grid(True, alpha=0.3, linestyle='--', axis='y')
        
        # Set tick font size
        self.ax.tick_params(axis='both', which='major', labelsize=self.tick_fontsize)
        
        self.fig.tight_layout()
        self.draw()
    
    def plot_enrichment_curve(self, scores, labels, gene_name="", lower_is_better=True):
        """Plot enrichment curve with enhanced fonts"""
        self.ax.clear()
        
        # Calculate enrichment at different fractions
        fractions = np.linspace(0.001, 0.2, 100)
        
        if lower_is_better:
            rank_scores = -np.array(scores)
        else:
            rank_scores = np.array(scores)
            
        sorted_indices = np.argsort(rank_scores)[::-1]
        sorted_labels = np.array(labels)[sorted_indices]
        
        n_total = len(labels)
        n_actives = np.sum(labels)
        
        enrichment = []
        
        for frac in fractions:
            n_top = max(1, int(n_total * frac))
            if n_top > 0 and n_actives > 0:
                n_actives_top = np.sum(sorted_labels[:n_top])
                expected = n_top * (n_actives / n_total)
                ef = n_actives_top / expected if expected > 0 else 0
                enrichment.append(ef)
            else:
                enrichment.append(0)
        
        # Plot enrichment curve
        self.ax.plot(fractions, enrichment, linewidth=3, color='red', label='Enrichment')
        self.ax.axhline(y=1, color='black', linestyle='--', alpha=0.5, label='Random', linewidth=2)
        
        # Set labels with larger fonts
        self.ax.set_xlabel('Fraction of Database Screened', fontsize=self.label_fontsize, fontweight='bold')
        self.ax.set_ylabel('Enrichment Factor', fontsize=self.label_fontsize, fontweight='bold')
        
        # Set title
        if gene_name:
            self.ax.set_title(f'Enrichment Curve - {gene_name}', fontsize=self.title_fontsize, fontweight='bold', pad=20)
        else:
            self.ax.set_title('Enrichment Curve', fontsize=self.title_fontsize, fontweight='bold', pad=20)
            
        # Configure legend
        self.ax.legend(loc='upper right', fontsize=self.legend_fontsize, framealpha=0.9)
        
        # Configure grid
        self.ax.grid(True, alpha=0.3, linestyle='--')
        
        # Set axis limits
        self.ax.set_xlim([0, 0.2])
        if len(enrichment) > 0:
            max_ef = max(enrichment)
            self.ax.set_ylim([0, max(20, max_ef * 1.1)])
        else:
            self.ax.set_ylim([0, 20])
            
        # Set tick font size
        self.ax.tick_params(axis='both', which='major', labelsize=self.tick_fontsize)
        
        self.fig.tight_layout()
        self.draw()
    
    def plot_bedroc_curve(self, scores, labels, alpha=20.0, gene_name="", lower_is_better=True):
        """Plot BEDROC early recognition curve with enhanced fonts"""
        self.ax.clear()
        
        if lower_is_better:
            rank_scores = -np.array(scores)
        else:
            rank_scores = np.array(scores)
            
        labels = np.array(labels)
        
        sorted_indices = np.argsort(rank_scores)[::-1]
        sorted_labels = labels[sorted_indices]
        
        n = len(labels)
        if n == 0:
            return
        
        ranks = np.arange(1, n + 1)
        r = ranks / n
        weights = np.exp(-alpha * r)
        hit_weights = weights * sorted_labels
        cum_weight = np.cumsum(hit_weights)
        
        total = cum_weight[-1] if cum_weight.size else 0.0
        if total > 0:
            cum_weight = cum_weight / total
        
        bed_val = ValidationMetrics.calculate_bedroc(scores, labels, alpha, lower_is_better)
        
        label_text = f'BEDROC(α={alpha:g}) = {bed_val.get("BEDROC", 0):.3f}' if bed_val.get("BEDROC", 0) > 0 else 'BEDROC - Not applicable'
        
        # Plot BEDROC curve
        self.ax.plot(ranks/n, cum_weight, lw=3, label=label_text, color='green')
        
        # Set labels with larger fonts
        self.ax.set_xlabel('Fraction Screened', fontsize=self.label_fontsize, fontweight='bold')
        self.ax.set_ylabel('Normalized Early-weighted Hit Recovery', fontsize=self.label_fontsize, fontweight='bold')
        
        # Set title
        if gene_name:
            self.ax.set_title(f'BEDROC Plot - {gene_name}', fontsize=self.title_fontsize, fontweight='bold', pad=20)
        else:
            self.ax.set_title('BEDROC Early Recognition Plot', fontsize=self.title_fontsize, fontweight='bold', pad=20)
            
        # Configure legend
        self.ax.legend(loc='lower right', fontsize=self.legend_fontsize, framealpha=0.9)
        
        # Configure grid
        self.ax.grid(True, alpha=0.3, linestyle='--')
        
        # Set tick font size
        self.ax.tick_params(axis='both', which='major', labelsize=self.tick_fontsize)
        
        self.fig.tight_layout()
        self.draw()

# ==================== MAIN GUI APPLICATION ====================

class DockingValidationForYourData(QMainWindow):
    """Main application window - Customized for your specific data"""
    
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Docking Validation Toolkit v2.1 - Customized for Your Affinity Data")
        self.setGeometry(100, 100, 1600, 1000)
        
        # Initialize variables
        self.actives_file = None
        self.decoys_file = None
        self.results = None
        self.column_mapping = None
        
        # Settings
        self.settings = {
            'lower_is_better': True,  # For AutoDock Vina, lower (more negative) is better
            'ef_fractions': [0.01, 0.05, 0.10],
            'bedroc_alpha': 20.0
        }
        
        # Setup UI
        self.init_ui()
        
        # Set initial status
        self.status_bar.showMessage("Ready - Please load your Actives.csv and Decoys.csv files")
    
    def init_ui(self):
        """Initialize the user interface"""
        
        # Central widget
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Main layout
        main_layout = QVBoxLayout(central_widget)
        
        # Top panel: File selection
        top_panel = self.create_top_panel()
        main_layout.addWidget(top_panel)
        
        # Splitter for main content
        splitter = QSplitter(Qt.Horizontal)
        
        # Left panel: Controls and results
        left_widget = QWidget()
        left_layout = QVBoxLayout(left_widget)
        
        # Settings panel
        settings_panel = self.create_settings_panel()
        left_layout.addWidget(settings_panel)
        
        # Control panel
        control_panel = self.create_control_panel()
        left_layout.addWidget(control_panel)
        
        # Results panel
        results_panel = self.create_results_panel()
        left_layout.addWidget(results_panel)
        
        # Right panel: Plots
        right_widget = QWidget()
        right_layout = QVBoxLayout(right_widget)
        
        # Plot selection and controls
        plot_controls = self.create_plot_controls()
        right_layout.addWidget(plot_controls)
        
        # Plot area
        self.plot_canvas = PlotCanvas(self, width=8, height=6.5, dpi=100)
        self.plot_toolbar = NavigationToolbar(self.plot_canvas, self)
        right_layout.addWidget(self.plot_toolbar)
        right_layout.addWidget(self.plot_canvas)
        
        # Add widgets to splitter
        splitter.addWidget(left_widget)
        splitter.addWidget(right_widget)
        splitter.setSizes([600, 1000])
        
        main_layout.addWidget(splitter)
        
        # Status bar
        self.status_bar = self.statusBar()
        self.status_bar.showMessage("Ready")
        
        # Progress bar
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.status_bar.addPermanentWidget(self.progress_bar)
    
    def create_top_panel(self):
        """Create top panel with file selection"""
        panel = QGroupBox("Step 1: Load Your CSV Files")
        layout = QGridLayout()
        
        # Information label
        info_label = QLabel("Your files should have columns: 'Ligand' and 'Affinity (kcal/mol)'\n"
                           "NA values will be automatically handled. Ensure numeric values in affinity column.")
        info_label.setStyleSheet("background-color: #e8f4f8; padding: 8px; border-radius: 5px; font-size: 11pt;")
        info_label.setWordWrap(True)
        layout.addWidget(info_label, 0, 0, 1, 4)
        
        # Active compounds file
        self.actives_label = QLabel("Actives CSV (Known Binders):")
        self.actives_path_label = QLabel("No file selected")
        self.actives_path_label.setStyleSheet("color: #666666; font-style: italic; padding: 5px;")
        self.actives_btn = QPushButton("Browse Actives...")
        self.actives_btn.setStyleSheet("background-color: #4CAF50; color: white; padding: 8px; font-weight: bold;")
        self.actives_btn.clicked.connect(self.select_actives_file)
        
        layout.addWidget(self.actives_label, 1, 0)
        layout.addWidget(self.actives_path_label, 1, 1, 1, 2)
        layout.addWidget(self.actives_btn, 1, 3)
        
        # Decoy compounds file
        self.decoys_label = QLabel("Decoys CSV (Inactive/Decoy Compounds):")
        self.decoys_path_label = QLabel("No file selected")
        self.decoys_path_label.setStyleSheet("color: #666666; font-style: italic; padding: 5px;")
        self.decoys_btn = QPushButton("Browse Decoys...")
        self.decoys_btn.setStyleSheet("background-color: #2196F3; color: white; padding: 8px; font-weight: bold;")
        self.decoys_btn.clicked.connect(self.select_decoys_file)
        
        layout.addWidget(self.decoys_label, 2, 0)
        layout.addWidget(self.decoys_path_label, 2, 1, 1, 2)
        layout.addWidget(self.decoys_btn, 2, 3)
        
        # Gene/Target name
        self.gene_label = QLabel("Target Name (Optional):")
        self.gene_edit = QLineEdit()
        self.gene_edit.setPlaceholderText("e.g., Target_X, COVID_Mpro, Kinase_Y")
        layout.addWidget(self.gene_label, 3, 0)
        layout.addWidget(self.gene_edit, 3, 1, 1, 3)
        
        panel.setLayout(layout)
        return panel
    
    def create_settings_panel(self):
        """Create settings panel"""
        panel = QGroupBox("Step 2: Analysis Settings")
        layout = QGridLayout()
        
        # Score direction
        self.lower_better_check = QCheckBox("✓ Lower affinity score is better (AutoDock Vina)")
        self.lower_better_check.setChecked(True)
        self.lower_better_check.stateChanged.connect(self.update_settings)
        layout.addWidget(self.lower_better_check, 0, 0, 1, 2)
        
        # Information about your data
        info_label = QLabel("Your data shows affinity in kcal/mol. More negative = stronger binding.")
        info_label.setStyleSheet("color: #666666; font-style: italic; padding: 5px; font-size: 10pt;")
        layout.addWidget(info_label, 1, 0, 1, 2)
        
        # BEDROC alpha
        self.bedroc_label = QLabel("BEDROC α (early enrichment weight):")
        self.bedroc_spin = QDoubleSpinBox()
        self.bedroc_spin.setRange(0.1, 100.0)
        self.bedroc_spin.setValue(20.0)
        self.bedroc_spin.setSingleStep(5.0)
        self.bedroc_spin.setToolTip("Higher α emphasizes early recognition more")
        self.bedroc_spin.valueChanged.connect(self.update_settings)
        layout.addWidget(self.bedroc_label, 2, 0)
        layout.addWidget(self.bedroc_spin, 2, 1)
        
        # EF fraction
        self.ef_label = QLabel("Enrichment Factor % (top fraction):")
        self.ef_spin = QDoubleSpinBox()
        self.ef_spin.setRange(0.1, 50.0)
        self.ef_spin.setValue(1.0)
        self.ef_spin.setSingleStep(0.5)
        self.ef_spin.setSuffix("%")
        self.ef_spin.setToolTip("Percentage of top-ranked compounds to calculate EF")
        self.ef_spin.valueChanged.connect(self.update_settings)
        layout.addWidget(self.ef_label, 3, 0)
        layout.addWidget(self.ef_spin, 3, 1)
        
        panel.setLayout(layout)
        return panel
    
    def create_control_panel(self):
        """Create control panel with buttons"""
        panel = QGroupBox("Step 3: Run Analysis")
        layout = QHBoxLayout()
        
        # Map columns button
        self.map_btn = QPushButton("✓ Confirm Column Mapping")
        self.map_btn.setStyleSheet("""
            QPushButton {
                background-color: #FF9800;
                color: white;
                padding: 12px;
                border-radius: 5px;
                font-weight: bold;
                font-size: 11pt;
            }
            QPushButton:hover {
                background-color: #F57C00;
            }
            QPushButton:disabled {
                background-color: #cccccc;
                color: #666666;
            }
        """)
        self.map_btn.clicked.connect(self.map_columns)
        self.map_btn.setEnabled(False)
        
        # Run validation button
        self.run_btn = QPushButton("▶ Run Docking Validation")
        self.run_btn.setStyleSheet("""
            QPushButton {
                background-color: #4CAF50;
                color: white;
                font-weight: bold;
                padding: 12px;
                border-radius: 5px;
                font-size: 12pt;
            }
            QPushButton:hover {
                background-color: #45a049;
            }
            QPushButton:disabled {
                background-color: #cccccc;
                color: #666666;
            }
        """)
        self.run_btn.clicked.connect(self.run_validation)
        self.run_btn.setEnabled(False)
        
        # Export results button
        self.export_btn = QPushButton("💾 Export All Results")
        self.export_btn.setStyleSheet("""
            QPushButton {
                background-color: #2196F3;
                color: white;
                padding: 12px;
                border-radius: 5px;
                font-weight: bold;
                font-size: 11pt;
            }
            QPushButton:hover {
                background-color: #0b7dda;
            }
            QPushButton:disabled {
                background-color: #cccccc;
                color: #666666;
            }
        """)
        self.export_btn.clicked.connect(self.export_results)
        self.export_btn.setEnabled(False)
        
        # Clear button
        self.clear_btn = QPushButton("🗑️ Clear All")
        self.clear_btn.setStyleSheet("""
            QPushButton {
                background-color: #f44336;
                color: white;
                padding: 12px;
                border-radius: 5px;
                font-size: 11pt;
            }
            QPushButton:hover {
                background-color: #da190b;
            }
        """)
        self.clear_btn.clicked.connect(self.clear_all)
        
        layout.addWidget(self.map_btn)
        layout.addWidget(self.run_btn)
        layout.addWidget(self.export_btn)
        layout.addWidget(self.clear_btn)
        layout.addStretch()
        
        panel.setLayout(layout)
        return panel
    
    def create_results_panel(self):
        """Create results display panel"""
        panel = QTabWidget()
        
        # Summary tab
        summary_widget = QWidget()
        summary_layout = QVBoxLayout(summary_widget)
        
        self.summary_text = QTextEdit()
        self.summary_text.setReadOnly(True)
        self.summary_text.setStyleSheet("""
            QTextEdit {
                background-color: #f8f9fa;
                border: 1px solid #dee2e6;
                border-radius: 4px;
                padding: 10px;
                font-family: Arial, sans-serif;
                font-size: 11pt;
            }
        """)
        summary_layout.addWidget(self.summary_text)
        
        # Metrics tab
        metrics_widget = QWidget()
        metrics_layout = QVBoxLayout(metrics_widget)
        
        self.metrics_table = QTableWidget()
        self.metrics_table.setColumnCount(2)
        self.metrics_table.setHorizontalHeaderLabels(['Metric', 'Value'])
        self.metrics_table.horizontalHeader().setStretchLastSection(True)
        self.metrics_table.horizontalHeader().setSectionResizeMode(0, QHeaderView.Stretch)
        self.metrics_table.horizontalHeader().setSectionResizeMode(1, QHeaderView.ResizeToContents)
        metrics_layout.addWidget(self.metrics_table)
        
        # Data tab
        data_widget = QWidget()
        data_layout = QVBoxLayout(data_widget)
        
        self.data_table = QTableWidget()
        data_layout.addWidget(self.data_table)
        
        # Statistics tab
        stats_widget = QWidget()
        stats_layout = QVBoxLayout(stats_widget)
        
        self.stats_text = QTextEdit()
        self.stats_text.setReadOnly(True)
        self.stats_text.setStyleSheet("""
            QTextEdit {
                background-color: #fff8e1;
                border: 1px solid #ffd54f;
                border-radius: 4px;
                padding: 10px;
                font-family: Consolas, Monaco, monospace;
                font-size: 10pt;
            }
        """)
        stats_layout.addWidget(self.stats_text)
        
        # Add tabs
        panel.addTab(summary_widget, "📊 Summary")
        panel.addTab(metrics_widget, "📈 Detailed Metrics")
        panel.addTab(data_widget, "📋 Data Preview")
        panel.addTab(stats_widget, "🔬 Statistical Analysis")
        
        return panel
    
    def create_plot_controls(self):
        """Create plot controls panel"""
        panel = QGroupBox("Plot Controls")
        layout = QHBoxLayout()
        
        # Plot selection combo box
        self.plot_combo = QComboBox()
        self.plot_combo.addItems([
            "📈 ROC Curve",
            "📊 Score Distribution",
            "📉 Enrichment Curve",
            "🎯 BEDROC Plot"
        ])
        self.plot_combo.currentIndexChanged.connect(self.update_plot)
        
        # Save plot button
        self.save_plot_btn = QPushButton("💾 Save Current Plot as TIFF")
        self.save_plot_btn.clicked.connect(self.save_plot_as_tiff)
        self.save_plot_btn.setEnabled(False)
        
        layout.addWidget(QLabel("Plot Type:"))
        layout.addWidget(self.plot_combo)
        layout.addStretch()
        layout.addWidget(self.save_plot_btn)
        
        panel.setLayout(layout)
        return panel
    
    def select_actives_file(self):
        """Select actives CSV file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Select Actives CSV File (Known Binders)",
            "",
            "CSV Files (*.csv);;Text Files (*.txt);;All Files (*)"
        )
        
        if file_path:
            self.actives_file = file_path
            base_name = os.path.basename(file_path)
            self.actives_path_label.setText(f"✓ {base_name}")
            self.actives_path_label.setStyleSheet("color: #4CAF50; font-weight: bold; padding: 5px;")
            self.check_files_ready()
            
            # Auto-set target name if empty
            if not self.gene_edit.text().strip():
                # Try to extract from filename
                base_no_ext = os.path.splitext(base_name)[0]
                if 'active' in base_no_ext.lower():
                    target_name = base_no_ext.replace('actives', '').replace('active', '').strip('_')
                    if target_name:
                        self.gene_edit.setText(target_name)
    
    def select_decoys_file(self):
        """Select decoys CSV file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Select Decoys CSV File (Inactive/Decoy Compounds)",
            "",
            "CSV Files (*.csv);;Text Files (*.txt);;All Files (*)"
        )
        
        if file_path:
            self.decoys_file = file_path
            base_name = os.path.basename(file_path)
            self.decoys_path_label.setText(f"✓ {base_name}")
            self.decoys_path_label.setStyleSheet("color: #2196F3; font-weight: bold; padding: 5px;")
            self.check_files_ready()
    
    def check_files_ready(self):
        """Check if both files are loaded"""
        if self.actives_file and self.decoys_file:
            self.map_btn.setEnabled(True)
            self.status_bar.showMessage("✓ Both files loaded. Click 'Confirm Column Mapping' to proceed.")
    
    def map_columns(self):
        """Open column mapping dialog - Simplified for your specific files"""
        try:
            # Read column headers (first row only)
            actives_df = pd.read_csv(self.actives_file, nrows=0)
            decoys_df = pd.read_csv(self.decoys_file, nrows=0)
            
            actives_columns = list(actives_df.columns)
            decoys_columns = list(decoys_df.columns)
            
            # Get or set gene name
            gene_name = self.gene_edit.text().strip()
            if not gene_name:
                gene_name = "Target_1"
                self.gene_edit.setText(gene_name)
            
            # Open mapping dialog
            dialog = ColumnMappingDialog(actives_columns, decoys_columns, gene_name, self)
            if dialog.exec_() == QDialog.Accepted:
                self.column_mapping = dialog.get_mapping()
                self.run_btn.setEnabled(True)
                self.status_bar.showMessage("✓ Columns mapped. Ready to run analysis!")
                
                # Show mapping confirmation
                mapping_info = f"""
                Column Mapping Confirmed:
                • Target Name: {self.column_mapping['gene_name']}
                • Actives ID: {self.column_mapping['actives_id']}
                • Actives Score: {self.column_mapping['actives_score']}
                • Decoys ID: {self.column_mapping['decoys_id']}
                • Decoys Score: {self.column_mapping['decoys_score']}
                
                Click 'Run Docking Validation' to start analysis.
                """
                self.summary_text.setText(mapping_info)
                
        except Exception as e:
            QMessageBox.critical(self, "Error", 
                f"Failed to read file headers:\n{str(e)}\n\n"
                f"Please ensure your CSV files have proper headers:\n"
                f"• 'Ligand' for compound names\n"
                f"• 'Affinity (kcal/mol)' for docking scores")
    
    def update_settings(self):
        """Update analysis settings"""
        self.settings['lower_is_better'] = self.lower_better_check.isChecked()
        self.settings['bedroc_alpha'] = self.bedroc_spin.value()
        ef_fraction = self.ef_spin.value() / 100.0
        self.settings['ef_fractions'] = [ef_fraction, 0.05, 0.10]
    
    def run_validation(self):
        """Run validation calculations"""
        if not self.actives_file or not self.decoys_file or not self.column_mapping:
            QMessageBox.warning(self, "Warning", 
                "Please complete all steps:\n"
                "1. Load both CSV files\n"
                "2. Confirm column mapping\n"
                "3. Run analysis")
            return
        
        # Disable buttons during calculation
        self.run_btn.setEnabled(False)
        self.map_btn.setEnabled(False)
        self.progress_bar.setVisible(True)
        self.progress_bar.setValue(0)
        self.status_bar.showMessage("Running docking validation analysis...")
        
        # Show progress in summary
        self.summary_text.setText("🔄 Running analysis...\n\n"
                                 "Processing steps:\n"
                                 "1. Loading and merging data\n"
                                 "2. Calculating ROC-AUC\n"
                                 "3. Computing enrichment factors\n"
                                 "4. Statistical analysis\n"
                                 "5. Generating plots\n\n"
                                 "This may take a moment...")
        
        # Create and start worker thread
        self.worker = ValidationWorker(self.actives_file, self.decoys_file, 
                                       self.column_mapping, self.settings)
        self.worker.progress.connect(self.update_progress)
        self.worker.result.connect(self.handle_results)
        self.worker.error.connect(self.handle_error)
        self.worker.finished.connect(self.calculation_finished)
        self.worker.start()
    
    def update_progress(self, value):
        """Update progress bar"""
        self.progress_bar.setValue(value)
    
    def handle_results(self, results):
        """Handle validation results"""
        self.results = results
        self.display_results()
        
        # Enable buttons
        self.export_btn.setEnabled(True)
        self.save_plot_btn.setEnabled(True)
        self.run_btn.setEnabled(True)
        self.map_btn.setEnabled(True)
        
        # Update plot
        self.update_plot()
        
        self.status_bar.showMessage("✓ Analysis complete! Ready to export results.")
        
        # Show success message
        QMessageBox.information(self, "Success", 
            f"Docking validation completed successfully!\n\n"
            f"• Target: {results.get('gene_name', 'Unknown')}\n"
            f"• Total compounds: {results['n_total']}\n"
            f"• Actives: {results['n_actives']}\n"
            f"• Decoys: {results['n_decoys']}\n"
            f"• ROC-AUC: {results['ROC_AUC']:.3f}\n\n"
            f"You can now:\n"
            f"1. View different plots using the dropdown\n"
            f"2. Save individual plots as TIFF (600 DPI)\n"
            f"3. Export all results using the export button")
    
    def handle_error(self, error_msg):
        """Handle calculation errors"""
        QMessageBox.critical(self, "Analysis Error", 
            f"Validation failed:\n\n{error_msg}\n\n"
            f"Common issues:\n"
            f"• Check if 'Affinity (kcal/mol)' column contains only numbers\n"
            f"• Remove any 'NA' or non-numeric values\n"
            f"• Ensure both files have the same column structure")
        
        # Reset buttons
        self.run_btn.setEnabled(True)
        self.map_btn.setEnabled(True)
        
        self.status_bar.showMessage("✗ Analysis failed. Please check your data.")
    
    def calculation_finished(self):
        """Clean up after calculation"""
        self.progress_bar.setVisible(False)
    
    def display_results(self):
        """Display validation results"""
        if not self.results:
            return
        
        # Update summary
        summary = self.generate_summary()
        self.summary_text.setHtml(summary)
        
        # Update metrics table
        self.update_metrics_table()
        
        # Update data table
        self.update_data_table()
        
        # Update statistics text
        self.update_statistics_text()
    
    def generate_summary(self):
        """Generate HTML summary of results"""
        gene_name = self.results.get('gene_name', 'Unknown Target')
        
        # Color coding based on performance
        roc_auc = self.results['ROC_AUC']
        if roc_auc >= 0.8:
            auc_color = "#2ecc71"  # Green
            auc_rating = "Excellent"
        elif roc_auc >= 0.7:
            auc_color = "#f1c40f"  # Yellow
            auc_rating = "Good"
        elif roc_auc >= 0.6:
            auc_color = "#e67e22"  # Orange
            auc_rating = "Moderate"
        else:
            auc_color = "#e74c3c"  # Red
            auc_rating = "Poor"
        
        ef_key = f'EF_{int(self.settings["ef_fractions"][0]*100)}%'
        ef_value = self.results.get(ef_key, 0)
        
        html = f"""
        <html>
        <head>
        <style>
            body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 15px; }}
            h1 {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; font-size: 18pt; }}
            h2 {{ color: #34495e; margin-top: 20px; border-left: 4px solid #3498db; padding-left: 10px; font-size: 14pt; }}
            .metric {{ margin: 12px 0; line-height: 1.6; }}
            .label {{ font-weight: bold; color: #2c3e50; display: inline-block; width: 250px; font-size: 11pt; }}
            .value {{ color: #27ae60; font-weight: bold; font-size: 11pt; }}
            .warning {{ color: #e74c3c; }}
            .success {{ color: #2ecc71; }}
            .info {{ color: #3498db; }}
            .section {{ background-color: #f8f9fa; padding: 15px; border-radius: 8px; margin: 12px 0; border-left: 5px solid #3498db; }}
            .highlight {{ background-color: #e8f4fd; padding: 5px 10px; border-radius: 4px; font-weight: bold; }}
            .rating {{ display: inline-block; padding: 3px 8px; border-radius: 4px; color: white; font-weight: bold; }}
            .note {{ font-style: italic; color: #7f8c8d; margin-top: 5px; font-size: 10pt; }}
            .stat {{ font-family: 'Consolas', monospace; }}
        </style>
        </head>
        <body>
        <h1>📊 Docking Validation Report - {gene_name}</h1>
        
        <div class="section">
        <h2>📋 Dataset Overview</h2>
        <div class="metric"><span class="label">Target Name:</span> <span class="highlight">{gene_name}</span></div>
        <div class="metric"><span class="label">Active Compounds:</span> <span class="value">{self.results['n_actives']} compounds</span></div>
        <div class="metric"><span class="label">Decoy Compounds:</span> <span class="value">{self.results['n_decoys']} compounds</span></div>
        <div class="metric"><span class="label">Total Database Size:</span> <span class="value">{self.results['n_total']} compounds</span></div>
        <div class="metric"><span class="label">Active Fraction:</span> <span class="value">{self.results['Active_Fraction']:.2%}</span></div>
        <div class="metric"><span class="label">Score Interpretation:</span> <span class="value">{"Lower affinity = better binding" if self.results['lower_is_better'] else "Higher affinity = better binding"}</span></div>
        </div>
        
        <div class="section">
        <h2>🎯 Key Performance Metrics</h2>
        <div class="metric">
            <span class="label">ROC-AUC Score:</span> 
            <span class="value">{self.results['ROC_AUC']:.3f}</span> 
            <span class="rating" style="background-color: {auc_color}; margin-left: 10px;">{auc_rating}</span>
        </div>
        <div class="metric"><span class="label">95% Confidence Interval:</span> <span class="value">{self.results['ROC_AUC_CI'][0]:.3f} - {self.results['ROC_AUC_CI'][1]:.3f}</span></div>
        <div class="metric"><span class="label">Enrichment Factor @1%:</span> <span class="value">{self.results.get('EF_1%', 0):.2f}x</span> <span class="note">(higher is better)</span></div>
        <div class="metric"><span class="label">Enrichment Factor @5%:</span> <span class="value">{self.results.get('EF_5%', 0):.2f}x</span></div>
        <div class="metric"><span class="label">Enrichment Factor @10%:</span> <span class="value">{self.results.get('EF_10%', 0):.2f}x</span></div>
        <div class="metric"><span class="label">BEDROC (α={self.results['alpha']:g}):</span> <span class="value">{self.results['BEDROC']:.3f}</span> <span class="note">(emphasizes early recognition)</span></div>
        </div>
        
        <div class="section">
        <h2>📈 Statistical Significance</h2>
        <div class="metric">
            <span class="label">Mann-Whitney U Test:</span> 
            <span class="value {'success' if self.results['MannWhitney_p'] < 0.05 else 'warning'}">{self.results['MannWhitney_p']:.2e}</span>
            <span class="rating" style="background-color: {'#2ecc71' if self.results['MannWhitney_p'] < 0.05 else '#e74c3c'}; margin-left: 10px;">
                {"Significant (p < 0.05)" if self.results['MannWhitney_p'] < 0.05 else "Not Significant"}
            </span>
        </div>
        <div class="metric"><span class="label">Cohen's d Effect Size:</span> <span class="value">{abs(self.results['Cohens_d']):.3f}</span> <span class="note">(|d| > 0.8 = large effect)</span></div>
        <div class="metric"><span class="label">Active Mean Score:</span> <span class="value">{self.results['Active_mean']:.3f} ± {self.results['Active_std']:.3f}</span></div>
        <div class="metric"><span class="label">Decoy Mean Score:</span> <span class="value">{self.results['Decoy_mean']:.3f} ± {self.results['Decoy_std']:.3f}</span></div>
        </div>
        
        <div class="section">
        <h2>🏆 Rank-Based Performance</h2>
        <div class="metric"><span class="label">Best Active Rank:</span> <span class="value">#{int(self.results['Best_Active_Rank'])}</span> <span class="note">(1 = best possible)</span></div>
        <div class="metric"><span class="label">Average Active Rank:</span> <span class="value">{self.results['Avg_Active_Rank']:.1f}</span> <span class="note">(out of {self.results['n_total']} total)</span></div>
        <div class="metric"><span class="label">Median Active Rank:</span> <span class="value">{self.results['Median_Active_Rank']:.1f}</span></div>
        <div class="metric"><span class="label">Worst Active Rank:</span> <span class="value">#{int(self.results['Worst_Active_Rank'])}</span></div>
        </div>
        
        <div class="section">
        <h2>💡 Interpretation Guidelines</h2>
        """
        
        # ROC-AUC interpretation
        if roc_auc >= 0.8:
            html += '<div class="metric success">✓ Excellent discrimination ability - Model can reliably distinguish actives from decoys</div>'
        elif roc_auc >= 0.7:
            html += '<div class="metric success">✓ Good discrimination ability - Model performs well at distinguishing actives from decoys</div>'
        elif roc_auc >= 0.6:
            html += '<div class="metric info">○ Moderate discrimination ability - Model has some ability to distinguish actives</div>'
        else:
            html += '<div class="metric warning">✗ Poor discrimination ability - Model cannot reliably distinguish actives from decoys</div>'
        
        # Early enrichment interpretation
        if ef_value >= 10:
            html += f'<div class="metric success">✓ Excellent early enrichment - {ef_value:.1f}x more actives found in top 1% than random</div>'
        elif ef_value >= 5:
            html += f'<div class="metric success">✓ Good early enrichment - {ef_value:.1f}x more actives found in top 1% than random</div>'
        elif ef_value >= 2:
            html += f'<div class="metric info">○ Moderate early enrichment - {ef_value:.1f}x more actives found in top 1% than random</div>'
        elif ef_value > 0:
            html += f'<div class="metric warning">✗ Poor early enrichment - Only {ef_value:.1f}x more actives found in top 1% than random</div>'
        
        # Statistical significance
        if self.results['MannWhitney_p'] < 0.05:
            html += '<div class="metric success">✓ Statistically significant difference between active and decoy scores</div>'
        else:
            html += '<div class="metric warning">✗ No statistically significant difference between active and decoy scores</div>'
        
        html += """
        </div>
        </body>
        </html>
        """
        
        return html
    
    def update_metrics_table(self):
        """Update the metrics table"""
        if not self.results:
            return
        
        # Prepare metrics data in categories
        categories = {
            "Dataset Information": [
                ("Target Name", self.results.get('gene_name', 'Unknown')),
                ("Total Compounds", f"{self.results['n_total']}"),
                ("Actives", f"{self.results['n_actives']}"),
                ("Decoys", f"{self.results['n_decoys']}"),
                ("Active Fraction", f"{self.results['Active_Fraction']:.4f}"),
            ],
            "ROC Metrics": [
                ("ROC-AUC", f"{self.results['ROC_AUC']:.4f}"),
                ("95% CI Lower", f"{self.results['ROC_AUC_CI'][0]:.4f}"),
                ("95% CI Upper", f"{self.results['ROC_AUC_CI'][1]:.4f}"),
                ("Sensitivity", f"{self.results['Sensitivity']:.4f}"),
                ("Specificity", f"{self.results['Specificity']:.4f}"),
            ],
            "Enrichment Factors": [
                (f"EF@{self.settings['ef_fractions'][0]*100:.0f}%", f"{self.results.get(f'EF_{int(self.settings['ef_fractions'][0]*100)}%', 0):.3f}"),
                ("EF@5%", f"{self.results.get('EF_5%', 0):.3f}"),
                ("EF@10%", f"{self.results.get('EF_10%', 0):.3f}"),
                (f"Hit Rate@{self.settings['ef_fractions'][0]*100:.0f}%", f"{self.results.get(f'HR_{int(self.settings['ef_fractions'][0]*100)}%', 0):.4f}"),
                (f"Yield@{self.settings['ef_fractions'][0]*100:.0f}%", f"{self.results.get(f'Yield_{int(self.settings['ef_fractions'][0]*100)}%', 0):.4f}"),
            ],
            "BEDROC Metrics": [
                (f"BEDROC (α={self.results['alpha']:g})", f"{self.results['BEDROC']:.4f}"),
                ("Normalized BEDROC", f"{self.results['BEDROC_norm']:.4f}"),
                ("RIE", f"{self.results['RIE']:.4f}"),
            ],
            "Statistical Tests": [
                ("Mann-Whitney p-value", f"{self.results['MannWhitney_p']:.2e}"),
                ("T-test p-value", f"{self.results['T_test_p']:.2e}"),
                ("KS-test p-value", f"{self.results['KS_p']:.2e}"),
                ("Cohen's d", f"{self.results['Cohens_d']:.4f}"),
            ],
            "Score Statistics": [
                ("Active Mean", f"{self.results['Active_mean']:.4f}"),
                ("Active Std Dev", f"{self.results['Active_std']:.4f}"),
                ("Decoy Mean", f"{self.results['Decoy_mean']:.4f}"),
                ("Decoy Std Dev", f"{self.results['Decoy_std']:.4f}"),
                ("Mean Difference", f"{self.results['Active_mean'] - self.results['Decoy_mean']:.4f}"),
            ],
            "Rank Statistics": [
                ("Best Active Rank", f"{int(self.results['Best_Active_Rank'])}"),
                ("Worst Active Rank", f"{int(self.results['Worst_Active_Rank'])}"),
                ("Average Active Rank", f"{self.results['Avg_Active_Rank']:.1f}"),
                ("Median Active Rank", f"{self.results['Median_Active_Rank']:.1f}"),
                ("GH Score @1%", f"{self.results['GH_Score_1%']:.4f}"),
                ("GH Score @5%", f"{self.results['GH_Score_5%']:.4f}"),
            ]
        }
        
        # Calculate total rows
        total_rows = sum(len(items) for items in categories.values()) + len(categories)  # + separators
        
        # Set up table
        self.metrics_table.setRowCount(total_rows)
        
        row = 0
        for category_name, items in categories.items():
            # Add category header
            header_item = QTableWidgetItem(f"── {category_name} ──")
            header_item.setBackground(QColor(52, 152, 219))
            header_item.setForeground(QColor(255, 255, 255))
            header_item.setFont(QFont("Arial", 10, QFont.Bold))
            self.metrics_table.setItem(row, 0, header_item)
            self.metrics_table.setItem(row, 1, QTableWidgetItem(""))
            self.metrics_table.setSpan(row, 0, 1, 2)
            row += 1
            
            # Add items
            for metric, value in items:
                self.metrics_table.setItem(row, 0, QTableWidgetItem(metric))
                self.metrics_table.setItem(row, 1, QTableWidgetItem(value))
                
                # Color code based on metric type
                if "p-value" in metric.lower():
                    try:
                        p_val = float(value.split('e')[0]) * 10 ** float(value.split('e')[1]) if 'e' in value else float(value)
                        if p_val < 0.05:
                            self.metrics_table.item(row, 1).setForeground(QColor(0, 128, 0))
                            self.metrics_table.item(row, 1).setText(f"✓ {value}")
                        elif p_val < 0.1:
                            self.metrics_table.item(row, 1).setForeground(QColor(255, 165, 0))
                        else:
                            self.metrics_table.item(row, 1).setForeground(QColor(255, 0, 0))
                    except:
                        pass
                elif "AUC" in metric or "EF" in metric or "BEDROC" in metric:
                    try:
                        val = float(value)
                        if val > 0.7:
                            self.metrics_table.item(row, 1).setForeground(QColor(0, 128, 0))
                        elif val > 0.6:
                            self.metrics_table.item(row, 1).setForeground(QColor(255, 165, 0))
                    except:
                        pass
                
                row += 1
        
        # Resize columns
        self.metrics_table.resizeColumnsToContents()
    
    def update_data_table(self):
        """Update the data table with loaded data"""
        if 'dataframe' not in self.results:
            return
        
        df = self.results['dataframe']
        
        # Show important columns
        display_cols = ['compound_id', 'docking_score', 'label']
        available_cols = [col for col in display_cols if col in df.columns]
        
        # Set up table
        max_rows = min(100, len(df))
        self.data_table.setRowCount(max_rows)
        self.data_table.setColumnCount(len(available_cols))
        self.data_table.setHorizontalHeaderLabels(['Ligand ID', 'Affinity (kcal/mol)', 'Type'])
        
        # Fill table
        for i in range(max_rows):
            for j, col in enumerate(available_cols):
                if col == 'compound_id':
                    value = str(df.iloc[i, df.columns.get_loc(col)])
                    item = QTableWidgetItem(value)
                elif col == 'docking_score':
                    value = df.iloc[i, df.columns.get_loc(col)]
                    item = QTableWidgetItem(f"{value:.3f}")
                    # Color code based on value
                    if self.results['lower_is_better']:
                        if value < self.results['Active_mean']:
                            item.setForeground(QColor(0, 128, 0))  # Green for good scores
                    else:
                        if value > self.results['Active_mean']:
                            item.setForeground(QColor(0, 128, 0))
                elif col == 'label':
                    value = df.iloc[i, df.columns.get_loc(col)]
                    display_value = "Active" if value == 1 else "Decoy"
                    item = QTableWidgetItem(display_value)
                    item.setForeground(QColor(0, 128, 0) if value == 1 else QColor(255, 0, 0))
                
                self.data_table.setItem(i, j, item)
        
        # Add note if more rows exist
        if len(df) > max_rows:
            self.data_table.setRowCount(max_rows + 1)
            note_item = QTableWidgetItem(f"... showing first {max_rows} of {len(df)} total compounds")
            note_item.setForeground(QColor(128, 128, 128))
            note_item.setFont(QFont("Arial", 9, QFont.StyleItalic))  # Fixed: QFont.StyleItalic
            self.data_table.setItem(max_rows, 0, note_item)
            self.data_table.setSpan(max_rows, 0, 1, len(available_cols))
        
        # Resize columns
        self.data_table.resizeColumnsToContents()
    
    def update_statistics_text(self):
        """Update the statistics text box"""
        if not self.results:
            return
        
        try:
            stats_text = f"""
            ============================================
            DOCKING VALIDATION - STATISTICAL ANALYSIS
            ============================================
            
            TARGET: {self.results.get('gene_name', 'Unknown')}
            DATE: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
            
            --------------------------------
            1. DATASET CHARACTERISTICS
            --------------------------------
            • Total compounds: {self.results['n_total']}
            • Active compounds: {self.results['n_actives']} ({self.results['Active_Fraction']:.2%})
            • Decoy compounds: {self.results['n_decoys']} ({1-self.results['Active_Fraction']:.2%})
            
            • Active mean score: {self.results['Active_mean']:.3f} ± {self.results['Active_std']:.3f}
            • Decoy mean score: {self.results['Decoy_mean']:.3f} ± {self.results['Decoy_std']:.3f}
            • Score difference: {self.results['Active_mean'] - self.results['Decoy_mean']:.3f}
            
            --------------------------------
            2. DISCRIMINATION PERFORMANCE
            --------------------------------
            ROC-AUC ANALYSIS:
            • ROC-AUC: {self.results['ROC_AUC']:.4f}
            • 95% Confidence Interval: [{self.results['ROC_AUC_CI'][0]:.4f}, {self.results['ROC_AUC_CI'][1]:.4f}]
            • Standard Error: {self.results['ROC_AUC_SE']:.4f}
            
            Interpretation:
            • AUC = 0.5: No discrimination (random)
            • AUC = 0.7-0.8: Acceptable discrimination
            • AUC = 0.8-0.9: Excellent discrimination
            • AUC > 0.9: Outstanding discrimination
            
            Current classification: {'EXCELLENT' if self.results['ROC_AUC'] >= 0.8 else 'GOOD' if self.results['ROC_AUC'] >= 0.7 else 'MODERATE' if self.results['ROC_AUC'] >= 0.6 else 'POOR'}
            
            --------------------------------
            3. EARLY ENRICHMENT METRICS
            --------------------------------
            ENRICHMENT FACTORS (EF):
            • EF@1%: {self.results.get('EF_1%', 0):.2f} (found {self.results.get('Actives_1%', 0)} actives in top 1%)
            • EF@5%: {self.results.get('EF_5%', 0):.2f} (found {self.results.get('Actives_5%', 0)} actives in top 5%)
            • EF@10%: {self.results.get('EF_10%', 0):.2f} (found {self.results.get('Actives_10%', 0)} actives in top 10%)
            
            BEDROC METRICS (Emphasizes early recognition):
            • BEDROC(α={self.results['alpha']:g}): {self.results['BEDROC']:.4f}
            • Normalized BEDROC: {self.results['BEDROC_norm']:.4f}
            • RIE: {self.results['RIE']:.4f}
            
            --------------------------------
            4. STATISTICAL SIGNIFICANCE
            --------------------------------
            HYPOTHESIS TESTS (H0: No difference between actives and decoys):
            
            • Mann-Whitney U Test (non-parametric):
              p-value = {self.results['MannWhitney_p']:.2e}
              Result: {'REJECT H0' if self.results['MannWhitney_p'] < 0.05 else 'FAIL TO REJECT H0'}
              Interpretation: {'Significant difference found (p < 0.05)' if self.results['MannWhitney_p'] < 0.05 else 'No significant difference (p ≥ 0.05)'}
            
            • Student's t-Test (parametric, unequal variance):
              p-value = {self.results['T_test_p']:.2e}
              Result: {'REJECT H0' if self.results['T_test_p'] < 0.05 else 'FAIL TO REJECT H0'}
            
            • Kolmogorov-Smirnov Test (distribution difference):
              p-value = {self.results['KS_p']:.2e}
              Result: {'REJECT H0' if self.results['KS_p'] < 0.05 else 'FAIL TO REJECT H0'}
            
            • Effect Size (Cohen's d):
              d = {self.results['Cohens_d']:.4f}
              Interpretation: {'LARGE effect (|d| > 0.8)' if abs(self.results['Cohens_d']) > 0.8 else 'MEDIUM effect (0.5 < |d| ≤ 0.8)' if abs(self.results['Cohens_d']) > 0.5 else 'SMALL effect (0.2 < |d| ≤ 0.5)' if abs(self.results['Cohens_d']) > 0.2 else 'NEGLIGIBLE effect (|d| ≤ 0.2)'}
            
            --------------------------------
            5. RANK-BASED ANALYSIS
            --------------------------------
            ACTIVE COMPOUND RANKS (1 = best):
            • Best rank: #{int(self.results['Best_Active_Rank'])}
            • Worst rank: #{int(self.results['Worst_Active_Rank'])}
            • Average rank: {self.results['Avg_Active_Rank']:.1f}
            • Median rank: {self.results['Median_Active_Rank']:.1f}
            
            GH SCORES (combines hit rate and yield):
            • GH@1%: {self.results['GH_Score_1%']:.4f}
            • GH@5%: {self.results['GH_Score_5%']:.4f}
            
            --------------------------------
            6. QUALITY ASSESSMENT
            --------------------------------
            OVERALL ASSESSMENT:
            """
            
            # Add assessment based on multiple metrics
            roc_auc = self.results['ROC_AUC']
            ef1 = self.results.get('EF_1%', 0)
            p_value = self.results['MannWhitney_p']
            
            if roc_auc >= 0.8 and ef1 >= 5 and p_value < 0.05:
                assessment = "EXCELLENT - Model shows strong discrimination and early enrichment with statistical significance"
            elif roc_auc >= 0.7 and ef1 >= 2 and p_value < 0.05:
                assessment = "GOOD - Model performs well with significant discrimination ability"
            elif roc_auc >= 0.6 and p_value < 0.05:
                assessment = "ACCEPTABLE - Model shows some discrimination ability"
            else:
                assessment = "POOR - Model cannot reliably distinguish actives from decoys"
            
            stats_text += f"• {assessment}\n\n"
            
            # Recommendations
            stats_text += """RECOMMENDATIONS:
            1. For virtual screening, prioritize EF@1% and BEDROC metrics
            2. Ensure statistical significance (p < 0.05) before proceeding
            3. Consider the trade-off between early enrichment (EF) and overall discrimination (AUC)
            4. For publication, report ROC-AUC with confidence intervals and EF@1%
            
            ============================================
            END OF REPORT
            ============================================
            """
            
            self.stats_text.setText(stats_text)
        except Exception as e:
            self.stats_text.setText(f"Error generating statistics: {str(e)}")
    
    def update_plot(self):
        """Update the plot based on selection"""
        if not self.results:
            return
        
        plot_type = self.plot_combo.currentText()
        gene_name = self.results.get('gene_name', '')
        
        if plot_type.startswith("📈 ROC"):
            self.plot_canvas.plot_roc_curve(
                self.results['FPR'], 
                self.results['TPR'], 
                self.results['ROC_AUC'],
                gene_name
            )
        elif plot_type.startswith("📊 Score"):
            self.plot_canvas.plot_score_distribution(
                self.results['active_scores'],
                self.results['decoy_scores'],
                gene_name,
                self.results['lower_is_better']
            )
        elif plot_type.startswith("📉 Enrichment"):
            self.plot_canvas.plot_enrichment_curve(
                self.results['scores'],
                self.results['labels'],
                gene_name,
                self.results['lower_is_better']
            )
        elif plot_type.startswith("🎯 BEDROC"):
            self.plot_canvas.plot_bedroc_curve(
                self.results['scores'],
                self.results['labels'],
                self.results['alpha'],
                gene_name,
                self.results['lower_is_better']
            )
    
    def save_plot_as_tiff(self):
        """Save the current plot as TIFF (600 dpi)"""
        if not self.results:
            QMessageBox.warning(self, "Warning", "No plot to save. Please run analysis first.")
            return
        
        gene_name = self.results.get('gene_name', 'Target')
        plot_type = self.plot_combo.currentText()
        
        # Clean filename
        safe_gene = re.sub(r'[^\w\-_]', '_', gene_name)
        
        # Map plot type to filename
        plot_map = {
            "📈 ROC Curve": "ROC_Curve",
            "📊 Score Distribution": "Score_Distribution",
            "📉 Enrichment Curve": "Enrichment_Curve",
            "🎯 BEDROC Plot": "BEDROC_Plot"
        }
        
        plot_key = plot_type
        plot_safe = plot_map.get(plot_key, plot_type.replace(' ', '_').replace('📈', '').replace('📊', '').replace('📉', '').replace('🎯', '').strip())
        
        default_filename = f"{safe_gene}_{plot_safe}.tiff"
        
        file_path, _ = QFileDialog.getSaveFileName(
            self,
            "Save Plot as High-Resolution TIFF",
            default_filename,
            "TIFF Files (*.tiff *.tif);;PNG Files (*.png);;PDF Files (*.pdf);;All Files (*)"
        )
        
        if file_path:
            try:
                # Determine format from extension
                if file_path.lower().endswith('.png'):
                    format = 'png'
                    dpi = 300
                elif file_path.lower().endswith('.pdf'):
                    format = 'pdf'
                    dpi = 300
                else:
                    # Default to TIFF
                    if not file_path.lower().endswith(('.tiff', '.tif')):
                        file_path += '.tiff'
                    format = 'tiff'
                    dpi = 600
                
                # Save with high DPI
                self.plot_canvas.fig.savefig(file_path, dpi=dpi, format=format, 
                                            bbox_inches='tight', pad_inches=0.1,
                                            facecolor='white', edgecolor='none')
                
                self.status_bar.showMessage(f"✓ Plot saved: {os.path.basename(file_path)}")
                QMessageBox.information(self, "Success", 
                    f"Plot saved as high-resolution image:\n\n{file_path}\n\n"
                    f"Image details:\n"
                    f"• Format: {format.upper()}\n"
                    f"• Resolution: {dpi} DPI\n"
                    f"• Size: ~{os.path.getsize(file_path) / 1024:.1f} KB\n"
                    f"• Suitable for publication")
                
            except Exception as e:
                QMessageBox.critical(self, "Error", 
                    f"Failed to save plot:\n\n{str(e)}\n\n"
                    f"Try:\n"
                    f"1. Different filename\n"
                    f"2. Different folder\n"
                    f"3. Check write permissions")
    
    def export_results(self):
        """Export all results (plots, data, summary)"""
        if not self.results:
            QMessageBox.warning(self, "Warning", "No results to export. Run analysis first.")
            return
        
        # Ask for directory
        directory = QFileDialog.getExistingDirectory(
            self,
            "Select Folder to Save All Results",
            "",
            QFileDialog.ShowDirsOnly
        )
        
        if not directory:
            return
        
        try:
            gene_name = self.results.get('gene_name', 'Target')
            safe_gene = re.sub(r'[^\w\-_]', '_', gene_name)
            timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
            
            # Create results subdirectory
            results_dir = os.path.join(directory, f"{safe_gene}_Results_{timestamp}")
            os.makedirs(results_dir, exist_ok=True)
            
            # Save all 4 plots as high-resolution images
            plot_types = ["📈 ROC Curve", "📊 Score Distribution", "📉 Enrichment Curve", "🎯 BEDROC Plot"]
            plot_filenames = []
            plot_names = []
            
            for plot_type in plot_types:
                # Update plot
                self.plot_combo.setCurrentText(plot_type)
                QApplication.processEvents()
                
                # Map to clean names
                name_map = {
                    "📈 ROC Curve": "ROC_Curve",
                    "📊 Score Distribution": "Score_Distribution",
                    "📉 Enrichment Curve": "Enrichment_Curve",
                    "🎯 BEDROC Plot": "BEDROC_Plot"
                }
                
                plot_safe = name_map.get(plot_type, plot_type.replace(' ', '_'))
                filename = f"{safe_gene}_{plot_safe}.tiff"
                file_path = os.path.join(results_dir, filename)
                
                # Save plot as high-resolution TIFF
                self.plot_canvas.fig.savefig(file_path, dpi=600, format='tiff', 
                                            bbox_inches='tight', pad_inches=0.1,
                                            facecolor='white', edgecolor='none')
                plot_filenames.append(filename)
                plot_names.append(plot_safe.replace('_', ' '))
            
            # Save summary as TXT
            summary_file = os.path.join(results_dir, f"{safe_gene}_Validation_Summary.txt")
            with open(summary_file, 'w', encoding='utf-8') as f:
                f.write("=" * 60 + "\n")
                f.write("DOCKING VALIDATION REPORT\n")
                f.write("=" * 60 + "\n\n")
                
                # Write basic info
                f.write(f"Target: {gene_name}\n")
                f.write(f"Analysis Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Total Compounds: {self.results['n_total']}\n")
                f.write(f"Actives: {self.results['n_actives']}\n")
                f.write(f"Decoys: {self.results['n_decoys']}\n")
                f.write(f"Active Fraction: {self.results['Active_Fraction']:.4f}\n\n")
                
                f.write("=" * 60 + "\n")
                f.write("KEY METRICS\n")
                f.write("=" * 60 + "\n\n")
                
                # Write key metrics
                f.write(f"ROC-AUC: {self.results['ROC_AUC']:.4f}\n")
                f.write(f"95% CI: [{self.results['ROC_AUC_CI'][0]:.4f}, {self.results['ROC_AUC_CI'][1]:.4f}]\n")
                f.write(f"EF@1%: {self.results.get('EF_1%', 0):.3f}\n")
                f.write(f"EF@5%: {self.results.get('EF_5%', 0):.3f}\n")
                f.write(f"EF@10%: {self.results.get('EF_10%', 0):.3f}\n")
                f.write(f"BEDROC: {self.results['BEDROC']:.4f}\n")
                f.write(f"Mann-Whitney p-value: {self.results['MannWhitney_p']:.2e}\n")
                f.write(f"Cohen's d: {self.results['Cohens_d']:.4f}\n\n")
                
                f.write("=" * 60 + "\n")
                f.write("STATISTICAL ANALYSIS\n")
                f.write("=" * 60 + "\n\n")
                
                # Write statistical analysis
                f.write(f"Active Mean Score: {self.results['Active_mean']:.4f} ± {self.results['Active_std']:.4f}\n")
                f.write(f"Decoy Mean Score: {self.results['Decoy_mean']:.4f} ± {self.results['Decoy_std']:.4f}\n")
                f.write(f"Score Difference: {self.results['Active_mean'] - self.results['Decoy_mean']:.4f}\n")
                f.write(f"T-test p-value: {self.results['T_test_p']:.2e}\n")
                f.write(f"KS-test p-value: {self.results['KS_p']:.2e}\n\n")
                
                f.write("=" * 60 + "\n")
                f.write("RANK STATISTICS\n")
                f.write("=" * 60 + "\n\n")
                
                f.write(f"Best Active Rank: {int(self.results['Best_Active_Rank'])}\n")
                f.write(f"Worst Active Rank: {int(self.results['Worst_Active_Rank'])}\n")
                f.write(f"Average Active Rank: {self.results['Avg_Active_Rank']:.1f}\n")
                f.write(f"Median Active Rank: {self.results['Median_Active_Rank']:.1f}\n\n")
                
                f.write("=" * 60 + "\n")
                f.write("Generated by Docking Validation Toolkit v2.1\n")
                f.write("=" * 60 + "\n")
            
            # Save metrics as CSV
            metrics_file = os.path.join(results_dir, f"{safe_gene}_Metrics.csv")
            
            # Prepare comprehensive metrics data
            metrics_data = []
            
            # Basic info
            metrics_data.append(("Target_Name", gene_name))
            metrics_data.append(("Analysis_Date", pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')))
            metrics_data.append(("Total_Compounds", self.results['n_total']))
            metrics_data.append(("Active_Compounds", self.results['n_actives']))
            metrics_data.append(("Decoy_Compounds", self.results['n_decoys']))
            metrics_data.append(("Active_Fraction", self.results['Active_Fraction']))
            
            # ROC metrics
            metrics_data.append(("ROC_AUC", self.results['ROC_AUC']))
            metrics_data.append(("ROC_AUC_CI_Lower", self.results['ROC_AUC_CI'][0]))
            metrics_data.append(("ROC_AUC_CI_Upper", self.results['ROC_AUC_CI'][1]))
            metrics_data.append(("ROC_AUC_SE", self.results['ROC_AUC_SE']))
            metrics_data.append(("Sensitivity", self.results['Sensitivity']))
            metrics_data.append(("Specificity", self.results['Specificity']))
            
            # Enrichment factors
            for fraction in self.settings['ef_fractions']:
                ef_key = f'EF_{int(fraction*100)}%'
                if ef_key in self.results:
                    metrics_data.append((ef_key, self.results[ef_key]))
            
            # BEDROC
            metrics_data.append((f"BEDROC_alpha_{self.results['alpha']:g}", self.results['BEDROC']))
            metrics_data.append(("BEDROC_Normalized", self.results['BEDROC_norm']))
            metrics_data.append(("RIE", self.results['RIE']))
            
            # Statistical tests
            metrics_data.append(("MannWhitney_p_value", self.results['MannWhitney_p']))
            metrics_data.append(("T_test_p_value", self.results['T_test_p']))
            metrics_data.append(("KS_test_p_value", self.results['KS_p']))
            metrics_data.append(("Cohens_d", self.results['Cohens_d']))
            
            # Score statistics
            metrics_data.append(("Active_Mean_Score", self.results['Active_mean']))
            metrics_data.append(("Active_Std_Dev", self.results['Active_std']))
            metrics_data.append(("Decoy_Mean_Score", self.results['Decoy_mean']))
            metrics_data.append(("Decoy_Std_Dev", self.results['Decoy_std']))
            
            # Rank statistics
            metrics_data.append(("Best_Active_Rank", self.results['Best_Active_Rank']))
            metrics_data.append(("Worst_Active_Rank", self.results['Worst_Active_Rank']))
            metrics_data.append(("Average_Active_Rank", self.results['Avg_Active_Rank']))
            metrics_data.append(("Median_Active_Rank", self.results['Median_Active_Rank']))
            
            # Create DataFrame and save
            metrics_df = pd.DataFrame(metrics_data, columns=['Metric', 'Value'])
            metrics_df.to_csv(metrics_file, index=False)
            
            # Save full data
            data_file = os.path.join(results_dir, f"{safe_gene}_Full_Data.csv")
            self.results['dataframe'].to_csv(data_file, index=False)
            
            # Save ROC curve data
            roc_file = os.path.join(results_dir, f"{safe_gene}_ROC_Curve_Data.csv")
            roc_df = pd.DataFrame({
                'FPR': self.results['FPR'],
                'TPR': self.results['TPR']
            })
            roc_df.to_csv(roc_file, index=False)
            
            # Save statistical analysis text
            stats_file = os.path.join(results_dir, f"{safe_gene}_Statistical_Analysis.txt")
            with open(stats_file, 'w', encoding='utf-8') as f:
                f.write(self.stats_text.toPlainText())
            
            self.status_bar.showMessage(f"✓ All results exported to: {results_dir}")
            
            # Show comprehensive success message
            message = f"""
            ✅ SUCCESS: All Results Exported!
            
            📁 Folder: {results_dir}
            
            📄 Files Created:
            
            1. High-Resolution Plots (600 DPI TIFF):
               • {plot_filenames[0]} - {plot_names[0]}
               • {plot_filenames[1]} - {plot_names[1]}
               • {plot_filenames[2]} - {plot_names[2]}
               • {plot_filenames[3]} - {plot_names[3]}
            
            2. Report Files:
               • {safe_gene}_Validation_Summary.txt
               • {safe_gene}_Metrics.csv
               • {safe_gene}_Full_Data.csv
               • {safe_gene}_ROC_Curve_Data.csv
               • {safe_gene}_Statistical_Analysis.txt
            
            💡 TIFF images are publication-ready (600 DPI)
            💡 CSV files contain all raw data for further analysis
            💡 TXT reports include comprehensive validation summary
            
            Total files: 9 files created successfully!
            """
            
            QMessageBox.information(self, "Export Complete", message)
            
        except Exception as e:
            QMessageBox.critical(self, "Export Error", 
                f"Failed to export results:\n\n{str(e)}\n\n"
                f"Possible issues:\n"
                f"• Write permissions in folder\n"
                f"• Disk space\n"
                f"• File already open\n\n"
                f"Try exporting to a different folder.")
    
    def clear_all(self):
        """Clear all data and reset the interface"""
        reply = QMessageBox.question(self, "Clear All", 
            "Are you sure you want to clear all data and reset the interface?\n\n"
            "This will remove:\n"
            "• Loaded files\n"
            "• Analysis results\n"
            "• All plots\n\n"
            "You cannot undo this action.",
            QMessageBox.Yes | QMessageBox.No, QMessageBox.No)
        
        if reply == QMessageBox.Yes:
            self.actives_file = None
            self.decoys_file = None
            self.results = None
            self.column_mapping = None
            
            # Reset UI elements
            self.gene_edit.clear()
            self.actives_path_label.setText("No file selected")
            self.actives_path_label.setStyleSheet("color: #666666; font-style: italic; padding: 5px;")
            self.decoys_path_label.setText("No file selected")
            self.decoys_path_label.setStyleSheet("color: #666666; font-style: italic; padding: 5px;")
            
            # Reset settings
            self.lower_better_check.setChecked(True)
            self.bedroc_spin.setValue(20.0)
            self.ef_spin.setValue(1.0)
            
            # Clear displays
            self.summary_text.clear()
            self.metrics_table.setRowCount(0)
            self.data_table.setRowCount(0)
            self.stats_text.clear()
            
            # Clear plot
            self.plot_canvas.ax.clear()
            self.plot_canvas.draw()
            
            # Disable buttons
            self.map_btn.setEnabled(False)
            self.run_btn.setEnabled(False)
            self.export_btn.setEnabled(False)
            self.save_plot_btn.setEnabled(False)
            
            self.status_bar.showMessage("✓ All data cleared. Ready to load new files.")

# ==================== APPLICATION ENTRY POINT ====================

def main():
    """Main application entry point"""
    # Enable high DPI scaling
    QApplication.setAttribute(Qt.AA_EnableHighDpiScaling, True)
    QApplication.setAttribute(Qt.AA_UseHighDpiPixmaps, True)
    
    # Create application
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    # Set application style with custom colors
    app.setStyleSheet("""
        QMainWindow {
            background-color: #f5f7fa;
        }
        QGroupBox {
            font-weight: bold;
            border: 2px solid #d1d9e6;
            border-radius: 8px;
            margin-top: 10px;
            padding-top: 15px;
            background-color: white;
            font-size: 11pt;
        }
        QGroupBox::title {
            subcontrol-origin: margin;
            left: 15px;
            padding: 0 10px 0 10px;
            color: #2c3e50;
            font-weight: bold;
        }
        QTableWidget {
            background-color: white;
            alternate-background-color: #f8fafc;
            gridline-color: #e2e8f0;
            border: 1px solid #e2e8f0;
            border-radius: 4px;
        }
        QTableWidget::item {
            padding: 5px;
        }
        QTableWidget::item:selected {
            background-color: #3498db;
            color: white;
        }
        QHeaderView::section {
            background-color: #2c3e50;
            color: white;
            padding: 8px;
            border: 1px solid #34495e;
            font-weight: bold;
        }
        QTextEdit {
            background-color: white;
            border: 1px solid #d1d9e6;
            border-radius: 4px;
            padding: 10px;
            font-family: 'Segoe UI', 'Arial', sans-serif;
        }
        QLineEdit, QComboBox, QSpinBox, QDoubleSpinBox {
            padding: 8px;
            border: 1px solid #d1d9e6;
            border-radius: 4px;
            background-color: white;
        }
        QLineEdit:focus, QComboBox:focus, QSpinBox:focus, QDoubleSpinBox:focus {
            border: 2px solid #3498db;
        }
        QPushButton {
            padding: 10px 15px;
            border-radius: 6px;
            font-weight: bold;
            border: none;
        }
        QPushButton:hover {
            opacity: 0.9;
        }
        QPushButton:pressed {
            opacity: 0.8;
        }
        QTabWidget::pane {
            border: 1px solid #d1d9e6;
            border-radius: 4px;
            background-color: white;
        }
        QTabBar::tab {
            background-color: #e2e8f0;
            padding: 10px 20px;
            margin-right: 2px;
            border-top-left-radius: 4px;
            border-top-right-radius: 4px;
        }
        QTabBar::tab:selected {
            background-color: #3498db;
            color: white;
        }
        QTabBar::tab:hover:!selected {
            background-color: #a0aec0;
        }
        QProgressBar {
            border: 1px solid #d1d9e6;
            border-radius: 4px;
            text-align: center;
            background-color: white;
        }
        QProgressBar::chunk {
            background-color: #3498db;
            border-radius: 3px;
        }
        QLabel {
            color: #2c3e50;
        }
        QCheckBox {
            spacing: 8px;
        }
        QCheckBox::indicator {
            width: 18px;
            height: 18px;
        }
    """)
    
    # Create and show main window
    window = DockingValidationForYourData()
    window.show()
    
    # Run application
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()

Actives loaded: 10 rows
Decoys loaded: 500 rows
Using columns - Actives ID: Ligand, Score: Affinity (kcal/mol)
Using columns - Decoys ID: Ligand, Score: Affinity (kcal/mol)
Actives after conversion: 10 rows
Decoys after conversion: 500 rows
NA values in Actives: 0
NA values in Decoys: 0
Actives after dropping NA: 10 rows
Decoys after dropping NA: 500 rows
Final combined dataset: 510 rows
Actives in final dataset: 10
Decoys in final dataset: 500


SystemExit: 0